# Interactive simulation checks: anemia screening

Verifies the IFA effect on hemoglobin, anemia-status assignment, hemoglobin-screening
coverage (baseline vs the `anemia_screening_vv` scenario), and that the hemoglobin test is
informative. Ported from the research portfolio VnV notebook
`model_18.3_interactive_simulation_anemia_screening`; updated to the current Engine
(`vivarium.engine`) API and to current model behavior.

Note: the source's `ifa_deleted_hemoglobin.exposure` / `first_anc_hemoglobin.exposure`
pipelines were removed, and the raw `hemoglobin_exposure` state column is not populated until
late in the timestep sequence -- so the IFA effect is expressed as IFA-vs-untreated
`hemoglobin.exposure`, and the sim is stepped to `delivery_facility` so the screening/anemia
columns are all populated. The exact test sensitivity/specificity targets (~0.85 / ~0.80) and
the precise hemoglobin measure the test screens are left for researchers to pin.

In [1]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [2]:
!pip list | grep vivarium

pytest-vivarium                          0.2.1


vivarium-artifact                        1.1.1
vivarium-build-utils                     4.8.0
vivarium-cluster-tools                   4.7.1
vivarium-config-tree                     5.2.1
vivarium-dependencies                    1.3.1
vivarium-engine                          5.10.1
vivarium-fuzzy-checker                   0.5.1
vivarium_gates_mncnh                     38.5.dev16+g472b4dd30 /mnt/share/homes/hjafari/repos/vgm_mic7462/vivarium_gates_mncnh
vivarium-gbd-mapping                     6.0.7
vivarium-public-health                   6.6.2
vivarium-risk-distributions              3.2.1



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
COLS = ["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
        "ferritin_screening_coverage", "tested_hemoglobin", "anemia_status_during_pregnancy",
        "hemoglobin.exposure"]

def build_sim(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    # Step through all ANC / screening events so the screening + anemia columns are populated.
    ev = sim._builder.time.simulation_event_name()
    while ev() != "delivery_facility":
        sim.step()
    return sim

def anemia_status_from(hb):
    return np.where(hb <= 70, "severe",
           np.where(hb <= 100, "moderate",
           np.where(hb <= 110, "mild", "not_anemic")))

In [4]:
# Baseline scenario
sim = build_sim()
df = sim.get_population(COLS)
df[["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
    "tested_hemoglobin", "anemia_status_during_pregnancy"]].head()

2026-09-23 14:45:32.247 | 0:00:16.815552 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for birth_outcome_probabilities. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:45:35.547 | 0:00:20.115119 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_all_causes.all_cause_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:45:35.614 | 0:00:20.181760 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:45:35.686 | 0:00:20.253941 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_preterm_birth_with_rds.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:45:35.750 | 0:00:20.317885 | WARNING  | simulation_1-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_preterm_birth_without_rds.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


...

2026-09-23 14:45:44.871 | 0:00:29.439269 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.postpartum_hemorrhage.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-09-23 14:45:44.872 | 0:00:29.439752 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.postpartum_hemorrhage.incidence_risk' configured, but didn't build lookup table 'tmred' during setup.


2026-09-23 14:45:44.872 | 0:00:29.440210 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.maternal_sepsis_and_other_maternal_infections.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-09-23 14:45:44.872 | 0:00:29.440632 | WARNING  | simulation_1-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.maternal_sepsis_and_other_maternal_infections.incidence_risk' configured, but didn't build lookup table 'tmred' during setup.


,anc_attendance,oral_iron_intervention,hemoglobin_screening_coverage,tested_hemoglobin,anemia_status_during_pregnancy
0,first_trimester_and_later_pregnancy,ifa,True,adequate,not_anemic
1,first_trimester_only,ifa,False,not_tested,<NA>
2,first_trimester_only,ifa,False,not_tested,<NA>
3,none,no_treatment,False,not_tested,<NA>
4,later_pregnancy_only,ifa,False,not_tested,not_anemic


## IFA raises hemoglobin

In [5]:
# The IFA effect is applied within the `hemoglobin.exposure` pipeline, so IFA-treated
# simulants have a higher hemoglobin exposure than the untreated.
ifa = df.oral_iron_intervention == "ifa"
assert df.loc[ifa, "hemoglobin.exposure"].mean() > df.loc[~ifa, "hemoglobin.exposure"].mean(), \
    "IFA-treated simulants do not have higher hemoglobin than the untreated"

## Anemia status is ordered by hemoglobin

In [6]:
# REVIEWER NOTE (loosened): exact threshold match (against the removed ifa_deleted_hemoglobin
# pipeline) replaced with an ordering-by-hemoglobin check.
# anemia_status_during_pregnancy is assigned to screened simulants from hemoglobin thresholds
# (severe <=70, moderate <=100, mild <=110, else not_anemic). Assert the assigned categories
# are ordered by mean hemoglobin.exposure.
known = ["severe", "moderate", "mild", "not_anemic"]
assigned = df[df.anemia_status_during_pregnancy.isin(known)]
assert len(assigned) > 0, "no simulants have an assigned anemia status"
order = assigned.groupby("anemia_status_during_pregnancy")["hemoglobin.exposure"].mean()
assert order["severe"] < order["moderate"] < order["mild"] < order["not_anemic"], \
    f"anemia-status categories not ordered by hemoglobin: {order.to_dict()}"

## Screening coverage: baseline

In [7]:
# Baseline hemoglobin screening happens at the later-pregnancy ANC visit, so only attendees
# with a later visit are (partially) screened; first-trimester-only and no-ANC are not; and
# ferritin screening is off at baseline.
cov = df.groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert cov.loc["none"] == 0, "hemoglobin screening occurred among no-ANC simulants at baseline"
later = ["later_pregnancy_only", "first_trimester_and_later_pregnancy"]
assert ((cov.loc[later] > 0) & (cov.loc[later] < 1)).all(), \
    f"expected partial baseline screening among later-visit ANC attendees, got {cov.to_dict()}"
assert cov.get("first_trimester_only", 0) == 0, \
    "first-trimester-only attendees were screened at baseline (screening is at the later visit)"
assert (~df.ferritin_screening_coverage).all(), "ferritin screening should be off at baseline"

## The hemoglobin test is informative

In [8]:
# REVIEWER NOTE (loosened): exact sensitivity/specificity targets (~0.85 / ~0.80, atol 0.05)
# relaxed to > 0.7; the truth basis (hemoglobin.exposure vs first_trimester_hemoglobin_exposure)
# is unconfirmed.
# Among screened simulants, the test should be well better than chance: most truly-low test
# low (sensitivity), most truly-adequate test adequate (specificity). Nominal targets are
# ~0.85 / ~0.80; truth is taken from hemoglobin.exposure (< 100 g/L). Researchers can tighten
# to exact targets and to whatever hemoglobin measure the test actually screens.
tested = df[df.tested_hemoglobin != "not_tested"].copy()
tested["truth"] = np.where(tested["hemoglobin.exposure"] < 100, "low", "adequate")
sens = (tested.loc[tested.truth == "low", "tested_hemoglobin"] == "low").mean()
spec = (tested.loc[tested.truth == "adequate", "tested_hemoglobin"] == "adequate").mean()
assert sens > 0.7, f"hemoglobin-test sensitivity {sens:.3f} unexpectedly low (target ~0.85)"
assert spec > 0.7, f"hemoglobin-test specificity {spec:.3f} unexpectedly low (target ~0.80)"

## Screening coverage: `anemia_screening_vv` scale-up scenario

In [9]:
# In the anemia-screening VnV scenario, later-pregnancy ANC attendees are all screened for
# hemoglobin and no-ANC simulants are still never screened.
vv = build_sim(scenario="anemia_screening_vv")
vv_cov = vv.get_population(["anc_attendance", "hemoglobin_screening_coverage"]) \
    .groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert vv_cov.loc["none"] == 0, "hemoglobin screening among no-ANC simulants in anemia_screening_vv"
later = ["later_pregnancy_only", "first_trimester_and_later_pregnancy"]
assert (vv_cov.loc[later] == 1).all(), \
    f"expected 100% hemoglobin screening at later-visit ANC in anemia_screening_vv, got {vv_cov.to_dict()}"

2026-09-23 14:46:20.239 | 0:01:04.807227 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:488 - Conflicting information for birth_outcome_probabilities. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:46:23.195 | 0:01:07.763106 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_all_causes.all_cause_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:46:23.286 | 0:01:07.854383 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:46:23.351 | 0:01:07.918844 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_preterm_birth_with_rds.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


2026-09-23 14:46:23.405 | 0:01:07.973205 | WARNING  | simulation_2-values_manager:_warn_if_overriding_resources:488 - Conflicting information for lbwsg_paf_on_neonatal_preterm_birth_without_rds.cause_specific_mortality_risk.paf. Ignoring 'required_resources' since the `source` is of type <class 'vivarium.engine.framework.lookup.table.LookupTable'> and we can infer the required resources directly.


...

2026-09-23 14:46:30.946 | 0:01:15.514234 | WARNING  | simulation_2-lookup_table_manager:on_post_setup:84 - Component 'risk_effect.low_birth_weight_and_short_gestation_on_cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk' configured, but didn't build lookup table 'relative_risk_interpolator' during setup.


2026-09-23 14:46:30.947 | 0:01:15.514788 | WARNING  | simulation_2-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.postpartum_hemorrhage.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-09-23 14:46:30.947 | 0:01:15.515371 | WARNING  | simulation_2-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.postpartum_hemorrhage.incidence_risk' configured, but didn't build lookup table 'tmred' during setup.


2026-09-23 14:46:30.948 | 0:01:15.515990 | WARNING  | simulation_2-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.maternal_sepsis_and_other_maternal_infections.incidence_risk' configured, but didn't build lookup table 'population_attributable_fraction' during setup.


2026-09-23 14:46:30.948 | 0:01:15.516516 | WARNING  | simulation_2-lookup_table_manager:on_post_setup:84 - Component 'non_log_linear_risk_effect.hemoglobin_on_cause.maternal_sepsis_and_other_maternal_infections.incidence_risk' configured, but didn't build lookup table 'tmred' during setup.
